In [ ]:
import pandas as pd
import numpy as np
import joblib
import io
import ipywidgets as widgets
from IPython.display import display, HTML

model = joblib.load('final_rf.joblib')

uploader = widgets.FileUpload(accept='.csv', multiple=False)
out = widgets.Output()

def on_upload_change(change):
    with out:
        out.clear_output()
        if uploader.value:
            try:
                uploaded_file = uploader.value[0] if isinstance(uploader.value, tuple) else list(uploader.value.values())[0]
                content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content
                
                df = pd.read_csv(io.BytesIO(content))
                features = list(model.feature_names_in_)
                
                metadata_candidates = ['Ref', 'Country', 'Region', 'Locality', 'Series', 'Sample']
                available_metadata = [col for col in metadata_candidates if col in df.columns]
                
                missing_features = [col for col in features if col not in df.columns]
                if missing_features:
                    print(f"Error: Missing required feature columns: {missing_features}")
                    return
                
                required_columns = available_metadata + features
                df_filtered = df[required_columns].copy()
                
                X = df_filtered[features].copy()
                X = X.fillna(1e-9)
                X = X.replace(0, 1e-9)
                
                X_log = np.log10(X)
                
                df_filtered['Tectonic_Setting'] = model.predict(X_log)
                df_filtered.to_csv('classified_results.csv', index=False)
                print("Classification successful. Download 'classified_results.csv' from the file directory panel.")
                
            except Exception as e:
                print(f"Error: {e}")

uploader.observe(on_upload_change, names='value')
display(HTML("<b>Upload your geochemical data CSV file:</b>"))
display(uploader, out)